<!-- xgate-data-path-note -->
> **Before running — data paths.** This notebook reads its inputs from a
> placeholder root `/path/to/xGATE/data/`. Replace `/path/to/xGATE` with the
> path to your local clone of this repository (or edit the paths to point at
> wherever you stored the data). See `configs/paths.example.yaml`,
> `data/README.md`, and `docs/data_availability.md` for the expected input
> files and where to obtain them. Some cells also *write* outputs under this
> root — adjust as needed.


In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '0' 
import random
random.seed(12)
import numpy as np
np.random.seed(12)
import torch
torch.manual_seed(12)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

import pandas as pd
import mygene
import sys, importlib
# Robust import for xGATE package (try 'xGATE' then 'xgate')
for _pkg in ("xGATE", "xgate"):
    try:
        utilities = importlib.import_module(f"{_pkg}.utilities")
        for _name in dir(utilities):
            if not _name.startswith("_"):
                globals()[_name] = getattr(utilities, _name)
        break
    except Exception:
        continue
else:
    raise ImportError("Could not import xGATE.utilities. Install the xGATE Python package (see README) or ensure it's on PYTHONPATH.")

test_pathways = [
    "Autophagy",
    "Protein processing in endoplasmic reticulum",
    "mTOR signaling pathway",
    "MAPK signaling pathway",
    "Apoptosis",
    "Longevity regulating pathway",
    "HIF-1 signaling pathway",
    "AMPK signaling pathway",
    "Phagosome",
    "PPAR signaling pathway",
    "FoxO signaling pathway",
    "cGMP-PKG signaling pathway",
    "Cytokine-cytokine receptor interaction",
    "Mineral absorption",
    "Antigen processing and presentation",
    "Rap1 signaling pathway",
    "Ribosome",
    "Protein digestion and absorption",
    "PI3K-Akt signaling pathway",
    "Insulin signaling pathway",
    "Oxidative phosphorylation",
    "Pancreatic secretion",
    "Salmonella infection",
    "Bacterial invasion of epithelial cells",
    "Pathogenic Escherichia coli infection",
    "Thyroid cancer",
    "Shigellosis"
]

print("T1D Pancreas Group")
# Load your count_matrix DataFrame
count_matrix = pd.read_csv("/path/to/xGATE/data/pancreas_t1d_final.csv", index_col=0)
count_filtered = count_matrix

row_means = np.mean(count_filtered, axis=1)

# Check if any row means are 0 or 1
rows_with_mean_zero_or_one = (row_means == 0) | (row_means == 1)

# If you need the indices of such rows
indices = np.where(rows_with_mean_zero_or_one)[0]

row_means = np.mean(count_filtered, axis=1)

# Identify the indices of rows with mean 0 or 1
rows_to_remove = (row_means == 0) | (row_means == 1)

# Remove these rows
count_filtered = count_filtered[~rows_to_remove]


# Uncomment these lines if adj_matrix is not already saved
'''
df = pd.DataFrame(count_filtered)
so = create_sifinet_object(df, rowfeature= True)
so = quantile_thres2(so)
so = cal_coexp(so, X = so.data_thres['dt'], X_full = so.data_thres['dt'])
so = create_network(so, alpha=0.05, manual=False, least_edge_prop=0.01)
so = filter_lowexp(so, t1=10, t2=0.9, t3=0.9)

adj_matrix = so.coexp


sif_ob_test = so

# Perform the element-wise comparison and assignment
adj_matrix = pd.DataFrame(np.where(
    np.abs(sif_ob_test.coexp - sif_ob_test.est_ms['mean']) > sif_ob_test.thres,
    np.abs(sif_ob_test.coexp),
    0
))

adj_matrix.index = df.index

adj_matrix.columns = df.index

adj_matrix = convert_gene_ids(adj_matrix, "ensembl")

adj_matrix.to_csv('/path/to/xGATE/data/adj_matrix_pancreas_t1d_final.csv', index=True)
'''
adj_matrix = pd.read_csv('/path/to/xGATE/data/adj_matrix_pancreas_t1d_final.csv', index_col=0)

G_s = utilities.create_network_from_adj_matrix(adj_matrix)

categorized_pathways = utilities.get_categorized_pathways()

pathway_results_t1d = utilities.analyze_pathways(G_s, test_pathways, categorized_pathways ,num_walks=200, max_walk_length = 200, null_dist_size = 200)

print(pathway_results_t1d)


In [ ]:
print("Control Pancreas Group")
# Load your count_matrix DataFrame
count_matrix = pd.read_csv("/path/to/xGATE/data/pancreas_ctrl_final.csv", index_col=0)
count_filtered = count_matrix

row_means = np.mean(count_filtered, axis=1)

# Check if any row means are 0 or 1
rows_with_mean_zero_or_one = (row_means == 0) | (row_means == 1)

# If you need the indices of such rows
indices = np.where(rows_with_mean_zero_or_one)[0]

row_means = np.mean(count_filtered, axis=1)

# Identify the indices of rows with mean 0 or 1
rows_to_remove = (row_means == 0) | (row_means == 1)

# Remove these rows
count_filtered = count_filtered[~rows_to_remove]


# Uncomment these lines if adj_matrix is not already saved
'''
df = pd.DataFrame(count_filtered)
so = create_sifinet_object(df, rowfeature= True)
so = quantile_thres2(so)
so = cal_coexp(so, X = so.data_thres['dt'], X_full = so.data_thres['dt'])
so = create_network(so, alpha=0.05, manual=False, least_edge_prop=0.01)
so = filter_lowexp(so, t1=10, t2=0.9, t3=0.9)

adj_matrix = so.coexp


sif_ob_test = so

# Perform the element-wise comparison and assignment
adj_matrix = pd.DataFrame(np.where(
    np.abs(sif_ob_test.coexp - sif_ob_test.est_ms['mean']) > sif_ob_test.thres,
    np.abs(sif_ob_test.coexp),
    0
))

adj_matrix.index = df.index

adj_matrix.columns = df.index

adj_matrix = convert_gene_ids(adj_matrix, "ensembl")

adj_matrix.to_csv('/path/to/xGATE/data/adj_matrix_pancreas_ctrl_final.csv', index=True)
'''

adj_matrix = pd.read_csv('/path/to/xGATE/data/adj_matrix_pancreas_ctrl_final.csv', index_col=0)

G_s = utilities.create_network_from_adj_matrix(adj_matrix)


categorized_pathways = utilities.get_categorized_pathways()

pathway_results_ctrl = utilities.analyze_pathways(G_s, test_pathways, categorized_pathways ,num_walks=200, max_walk_length = 200, null_dist_size = 200)
print(pathway_results_ctrl)

